# 02 Features: cleaning and feature engineering 
All feature creation is implemented in `src/features.py` and wrapped in a scikit-learn Pipeline with a ColumnTransformer. Learning steps (percentile caps, median fills, category lists, scaling) are fitted on the training split only.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../src")

model_data = pd.read_parquet("../data/processed/model_data.parquet")
train = model_data[model_data["split"] == "train"].copy()
valid = model_data[model_data["split"] == "valid"].copy()
test = model_data[model_data["split"] == "test"].copy()
print(train.shape, valid.shape, test.shape)

### Import the pipeline

In [ ]:
import sys
sys.path.append("../src")
from features import (engineer_features, get_X, build_preprocessing_pipeline,
                      NUMERIC_A, NUMERIC_B_EXTRA, CATEGORICAL, CAP_RULES)

#### Look at the new columns and check them

In [ ]:
eng = engineer_features(get_X(train, "B"))
new_cols = ["emp_length_years", "credit_history_years", "fico_mid", "log_annual_inc",
            "loan_to_income", "installment_to_income", "grade_num"]

print(eng[new_cols].describe().T.round(2).to_string())
print()
print("emp_length parsing check (original text -> average parsed number):")
print(eng.groupby(train["emp_length"].fillna("missing"))["emp_length_years"].mean())
print()
print("Negative credit history lengths:", (eng["credit_history_years"] < 0).sum())
print("Infinite values in ratios:", np.isinf(eng[["loan_to_income", "installment_to_income"]]).sum().to_dict())
print()
print("Missing values in new columns:")
print(eng[new_cols].isna().sum())

#### The revol_util above 100 check 

In [ ]:
over = train["revol_util"] > 100
print("Training loans with revol_util above 100:", over.sum(), f"({over.mean() * 100:.2f}%)")
print("Default rate when revol_util > 100:", round(train.loc[over, "default"].mean() * 100, 2), "% | overall:", round(train["default"].mean() * 100, 2), "%")
print("Largest value:", train["revol_util"].max())
print("Cap that will be applied (99.9th percentile of train):", round(train["revol_util"].quantile(0.999), 2))

#### Build, fit on train, and transform all three splits

In [ ]:
results = {}
for fs in ["A", "B"]:
    prep = build_preprocessing_pipeline(feature_set=fs, scale=False)
    X_train = prep.fit_transform(get_X(train, fs))     # FIT happens here, on TRAIN only
    X_valid = prep.transform(get_X(valid, fs))         # transform only: nothing is learned
    X_test = prep.transform(get_X(test, fs))           # transform only: nothing is learned
    results[fs] = (prep, X_train, X_valid, X_test)

    print(f"Set {fs}: train {X_train.shape} | valid {X_valid.shape} | test {X_test.shape}")
    print("   missing values left (train, valid, test):",
          int(X_train.isna().sum().sum()), int(X_valid.isna().sum().sum()), int(X_test.isna().sum().sum()))
    print("   all columns numeric:", X_train.select_dtypes("number").shape[1] == X_train.shape[1])
    print("   same columns in all splits:", list(X_train.columns) == list(X_valid.columns) == list(X_test.columns))

#### Inspect what the pipeline learned

In [ ]:
prep_A, XA_train, XA_valid, XA_test = results["A"]
prep_B, XB_train, XB_valid, XB_test = results["B"]

print("Missing-value indicator columns created:")
print([c for c in XA_train.columns if c.startswith("missingindicator_")])
print()

ohe = prep_A.named_steps["columns"].named_transformers_["cat"].named_steps["onehot"]
print("Rare categories merged into 'infrequent' (under 1% of training loans):")
for name, infreq in zip(CATEGORICAL, ohe.infrequent_categories_):
    print("  ", name, "->", list(infreq) if infreq is not None else "no rare categories")
print()

cap = prep_B.named_steps["columns"].named_transformers_["num"].named_steps["cap"]
bounds = pd.DataFrame(cap.bounds_, index=["lower", "upper"]).T.round(3)
eng_valid = engineer_features(get_X(valid, "B"))
bounds["valid_share_capped_%"] = [
    (((eng_valid[c] < lo) | (eng_valid[c] > hi)).mean() * 100).round(2) for c, (lo, hi) in cap.bounds_.items()
]
print("Cap boundaries learned from TRAIN, and the share of validation values they cap:")
print(bounds)

for fs in ["A", "B"]:
    pd.Series(list(results[fs][1].columns), name="feature").to_csv(f"../reports/final_features_{fs}.csv", index=False)
print()
print("Saved final feature lists to reports/final_features_A.csv and final_features_B.csv")

#### Proof that scaling is fitted on train only

In [ ]:
prep_scaled = build_preprocessing_pipeline(feature_set="A", scale=True)
Xs_train = prep_scaled.fit_transform(get_X(train, "A"))
Xs_valid = prep_scaled.transform(get_X(valid, "A"))

cols = ["dti", "fico_mid", "log_annual_inc"]
check = pd.DataFrame({
    "train_mean": Xs_train[cols].mean(), "train_std": Xs_train[cols].std(),
    "valid_mean": Xs_valid[cols].mean(), "valid_std": Xs_valid[cols].std(),
}).round(3)
print(check)

#### One more distribution check

In [ ]:
print(pd.crosstab(model_data["split"], model_data["application_type"]))

**Note:** joint applications appear only in the 2015 test split (239 loans, 0.08%). The model never sees them in training, so they are encoded as all zeros in the application-type columns. This is a minor limitation.

### Summary
**Engineered (row-wise, no learning):** `emp_length_years` (text to number), `credit_history_years` (earliest credit line to issue date), `fico_mid`, `log_annual_inc`, `loan_to_income`, `installment_to_income` (monthly payment / monthly income, set B only), `grade_num` (A=1 ... G=7, set B only).

**Learned on the training split only:** percentile caps for heavy-tailed features (1st/99th for log income, 99th for ratios, balances and account counts, 99.9th for revolving utilization), median fills with missing-value indicator columns, rare-category grouping (under 1% of training loans) and one-hot categories, and optional standard scaling (logistic regression only).

**Checks:** no missing values remain after the pipeline, all columns are numeric, train/validation/test have identical columns, and scaled validation means differ slightly from 0, confirming scaling was fitted on train only.

**Feature sets:** A (application-only) and B (A plus int_rate, grade_num, installment_to_income). Final column names are saved in `reports/final_features_A.csv` and `reports/final_features_B.csv`. The whole process lives in `src/features.py` and is reused unchanged in the modeling notebook.

**Note on revol_util:** values above 100 are real (borrowers over their credit limit), rare, and capped near the 99.9th percentile rather than removed.